In [16]:
# %pip install soundfile
!pip install -q kaggle

In [ ]:
# import soundfile as sf
# print(sf.__version__)

In [1]:
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
import json
import re
import wave
import numpy as np
import soundfile as sf
import os

import torch
from torch import Tensor, nn
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader, Dataset



## Download dataset
- The dataset is stored in the temporary Colab filesystem

In [3]:
%cd /content

!git clone --filter=blob:none --no-checkout \
    https://github.com/microsoft/AEC-Challenge.git

%cd /content/AEC-Challenge

!git sparse-checkout init --cone

!git sparse-checkout set \
    datasets/synthetic/nearend_mic_signal \
    datasets/synthetic/nearend_speech

!git checkout main

/content
Cloning into 'AEC-Challenge'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 345 (delta 24), reused 47 (delta 12), pack-reused 279 (from 1)
Receiving objects: 100% (345/345), 2.70 MiB | 2.70 MiB/s, done.
Resolving deltas: 100% (139/139), done.
/content/AEC-Challenge
remote: Enumerating objects: 20009, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 20009 (delta 0), reused 1 (delta 0), pack-reused 20005 (from 1)
Receiving objects: 100% (20009/20009), 3.38 MiB | 18.82 MiB/s, done.
Updating files: 100% (20009/20009), done.
Filtering content: 100% (20000/20000), 5.96 GiB | 5.42 MiB/s, done.
Already on 'main'
Your branch is up to date with 'origin/main'.


In [13]:
import json
from pathlib import Path

dataset_dir = Path("/content/erb_store")

metadata = {
    "title": "Speech Dataset",
    "id": "quanninhhoang/erb-speech-dataset",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(dataset_dir / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f)

In [19]:
!kaggle datasets create -p /content/erb_store --dir-mode zip

Starting upload for file shard_0018.pt
100% 1.02G/1.02G [00:48<00:00, 22.3MB/s]
Upload successful: shard_0018.pt (1GB)
Starting upload for file shard_0032.pt
100% 1.02G/1.02G [00:50<00:00, 21.7MB/s]
Upload successful: shard_0032.pt (1GB)
Starting upload for file shard_0040.pt
100% 1.02G/1.02G [00:51<00:00, 21.0MB/s]
Upload successful: shard_0040.pt (1GB)
Starting upload for file shard_0000.pt
100% 1.02G/1.02G [00:48<00:00, 22.4MB/s]
Upload successful: shard_0000.pt (1GB)
Starting upload for file shard_0035.pt
100% 1.02G/1.02G [00:51<00:00, 21.4MB/s]
Upload successful: shard_0035.pt (1GB)
Starting upload for file shard_0025.pt
100% 1.02G/1.02G [00:49<00:00, 21.9MB/s]
Upload successful: shard_0025.pt (1GB)
Starting upload for file shard_0043.pt
100% 1.02G/1.02G [00:49<00:00, 22.1MB/s]
Upload successful: shard_0043.pt (1GB)
Starting upload for file shard_0010.pt
100% 1.02G/1.02G [00:49<00:00, 21.9MB/s]
Upload successful: shard_0010.pt (1GB)
Starting upload for file shard_0009.pt
100% 1.02

In [18]:
import os
from kaggle.api.kaggle_api_extended import KaggleApi

os.environ["KAGGLE_API_TOKEN"] = "KGAT_1c5cfe7583e9b4ef3dc5b032a7a62ec1"

api = KaggleApi()
api.authenticate()

print("Kaggle authentication successful!")

Kaggle authentication successful!


## Verify dataset

In [ ]:
# input_dir = Path(
#     "/content/AEC-Challenge/datasets/synthetic/nearend_mic_signal"
# )

# target_dir = Path(
#     "/content/AEC-Challenge/datasets/synthetic/nearend_speech"
# )

# print(len(list(input_dir.glob("*.wav"))), "input files")
# print(len(list(target_dir.glob("*.wav"))), "target files")

10000 input files
10000 target files


## Mount google drive to server(e.g, GPU T4)

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## ERB filter bank

In [4]:
def load_mono(path):
    wav, sr = torchaudio.load(path)    # sr: sample rate
    wav = wav.mean(dim=0)
    return wav, sr

def stft_magnitude(wav, device, n_fft=512, hop_length=128, win_length=512):
    window = torch.hann_window(win_length, device=device)
    wav = wav.to(device)

    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True,
    )

    # [freq, time] -> [time, freq]
    return spec.transpose(0, 1)

def hz_to_erb(freq):
    return 21.4 * torch.log10(1.0 + 0.00437 * freq)

def erb_to_hz(erb):
    return (10 ** (erb / 21.4) - 1.0) / 0.00437

def make_erb_filterbank(sample_rate, n_fft, erb_bins, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sample_rate / 2

    erb_edges = torch.linspace(
        hz_to_erb(torch.tensor(low_freq)),
        hz_to_erb(torch.tensor(high_freq)),
        erb_bins + 2,
    )

    hz_edges = erb_to_hz(erb_edges)
    fft_freqs = torch.linspace(0.0, sample_rate / 2, n_fft // 2 + 1)

    filterbank = torch.zeros(erb_bins, n_fft // 2 + 1)

    for i in range(erb_bins):
        left = hz_edges[i]
        center = hz_edges[i + 1]
        right = hz_edges[i + 2]

        rising = (fft_freqs - left) / (center - left)
        falling = (right - fft_freqs) / (right - center)

        filterbank[i] = torch.minimum(rising, falling).clamp_min(0.0)
    return filterbank

# def wav_to_erb(path, sample_rate, filterbank, n_fft=512, hop_length=128, win_length=512):
#     wav, sr = load_mono(path)

#     if sr != sample_rate:
#         wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=sample_rate)

#     magnitude = stft_magnitude(wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
#     erb = magnitude @ filterbank.T
#     erb = torch.log1p(erb)
#     return erb 

def wav_pair_to_features(input_path: str | Path, target_path: str | Path, *, sample_rate: int, n_fft: int, hop_length: int, win_length: int, filterbank: Tensor, device) -> Dict[str, Tensor]:
    input_wav, input_sr = load_mono(input_path)
    target_wav, target_sr = load_mono(target_path)

    if input_sr != sample_rate:
        input_wav = torchaudio.functional.resample(input_wav, input_sr, sample_rate)
    if target_sr != sample_rate:
        target_wav = torchaudio.functional.resample(target_wav, target_sr, sample_rate)

    sample_count = min(input_wav.numel(), target_wav.numel())
    input_wav = input_wav[:sample_count]
    target_wav = target_wav[:sample_count]

    input_spec = stft_magnitude(input_wav, device, n_fft, hop_length, win_length)
    target_spec = stft_magnitude(target_wav, device, n_fft, hop_length, win_length)
    frame_count = min(input_spec.shape[0], target_spec.shape[0])
    input_spec = input_spec[:frame_count].contiguous()
    target_spec = target_spec[:frame_count].contiguous()

    input_power = input_spec.abs().square()
    target_power = target_spec.abs().square()
    input_erb = torch.log1p(input_power @ filterbank.to(device).T).float()
    target_erb = torch.log1p(target_power @ filterbank.to(device).T).float()

    record = {
        "input_erb": input_erb.cpu(),
        "target_erb": target_erb.cpu(),
        "input_spec": input_spec.to(torch.complex64).cpu(),
        "target_spec": target_spec.to(torch.complex64).cpu(),
    }

    del input_wav
    del target_wav
    del input_spec
    del target_spec
    del input_erb
    del target_erb

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return record

def build_erb_store(input_dir: str | Path, 
                    target_dir: str | Path, 
                    output_dir: str | Path, 
                    *, 
                    sample_rate: int = 16000, 
                    n_fft: int = 512, 
                    hop_length: int = 128, 
                    win_length: int = 512, 
                    erb_bins: int = 32,
                    device) -> None:
    input_dir = Path(input_dir)
    target_dir = Path(target_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    input_files = sorted(p for p in input_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".wav")
    target_files = sorted(p for p in target_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".wav")

    if not input_files:
        raise FileNotFoundError(f"No WAV files found under {input_dir}")
    if not target_files:
        raise FileNotFoundError(f"No WAV files found under {target_dir}")

    target_by_id: Dict[str, Path] = {}
    for target_path in target_files:
        match = re.search(r"(?:^|_)fileid_(.+)\.wav$", target_path.name)
        if match:
            target_by_id[match.group(1)] = target_path

    filterbank = make_erb_filterbank(sample_rate, n_fft, erb_bins)
    index_rows: List[Dict[str, Any]] = []
    shard_size = 200
    shard = []
    shard_id = 0

    for record_id, input_path in enumerate(input_files):
        if record_id % 10 == 0:
            print(
                f"Processing {record_id}/{len(input_files)}",
                flush=True,
            )
        target_path = target_dir / input_path.name
        if not target_path.exists():
            match = re.search(r"(?:^|_)fileid_(.+)\.wav$", input_path.name)
            target_path = target_by_id.get(match.group(1)) if match else None

        if target_path is None or not target_path.exists():
            raise FileNotFoundError(f"Missing target for {input_path.name} under {target_dir}")

        record = wav_pair_to_features(
            input_path,
            target_path,
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            filterbank=filterbank,
            device=device,
        )
        shard_index = len(shard)
        shard.append(record)

        index_rows.append(
            {
                "id": record_id,
                "input_file": str(input_path),
                "target_file": str(target_path),
                "shard_file": f"shard_{shard_id:04d}.pt",
                "shard_index": shard_index,
                "frames": int(record["input_erb"].shape[0]),
                "erb_bins": int(record["input_erb"].shape[1])
            }
        )

        if (len(shard) == shard_size):
            shard_path = (output_dir / f"shard_{shard_id:04d}.pt")
            torch.save(shard, shard_path)
            shard = []
            shard_id += 1
    if shard:
        shard_path = (output_dir / f"shard_{shard_id:04d}.pt")
        torch.save(shard, shard_path)
        
    index_path = output_dir / "index.jsonl"
    with index_path.open("w", encoding="utf-8") as handle:
        for row in index_rows:
            handle.write(json.dumps(row) + os.linesep)


## Build Dataset Object

In [5]:
class IndexedERBDataset(Dataset):
    def __init__(self, index_path: str | Path, *, segment_frames: Optional[int] = 256, random_crop: bool = True) -> None:
        index_path = Path(index_path)
        self.data_dir = index_path.parent
        self.rows = [
            json.loads(line)
            for line in Path(index_path).read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        self.segment_frames = segment_frames
        self.random_crop = random_crop

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> Dict[str, Tensor]:
        row = self.rows[index]
        shard_path = self.data_dir / row["shard_file"]
        shard = torch.load(shard_path, map_location="cpu", weights_only=True)
        record = shard[row["shard_index"]]
        total_frames = record["input_erb"].shape[0]
        frames = self.segment_frames or total_frames

        if total_frames >= frames:
            if self.random_crop:
                start = torch.randint(total_frames - frames + 1, ()).item()
            else:
                start = (total_frames - frames) // 2
            stop = start + frames
            result = {key: value[start:stop] for key, value in record.items()}
        else:
            result = {
                key: torch.nn.functional.pad(
                    value,
                    (0, 0, 0, frames - total_frames),
                )
                for key, value in record.items()
            }
        # Model input: [B, 1, T, E]; DataLoader adds B.
        result["input_erb"] = result["input_erb"].unsqueeze(0)
        result["target_erb"] = result["target_erb"].unsqueeze(0)
        return result

In [7]:
"""Return normalized inverse ERB weights with shape ``[F, E]``."""
def erb_synthesis_matrix(filterbank: Tensor, eps: float = 1e-8) -> Tensor:
    return filterbank.T / filterbank.sum(dim=0, keepdim=True).T.clamp_min(eps)

def apply_erb_gains(input_spec: Tensor, gains: Tensor, synthesis_matrix: Tensor) -> Tensor:
    if gains.ndim != 4 or gains.shape[1] != 1:
        raise ValueError("gains must have shape [B, 1, T, E]")
    """synthesis_matrix: [F, E], which converts values defined on 
    ERB bands back into values for individual FFT frequency bins.
       gains[:, 0]: makes [B, 1, T, E] -> [B, T, E]"""
    frequency_gain = gains[:, 0] @ synthesis_matrix.T    # -> [B, T, F]
    return input_spec * frequency_gain.to(input_spec.dtype)

def compressed_spectral_loss(predicted_spec: Tensor, target_spec: Tensor, power: float = 0.6) -> Tensor:
    pred_mag = predicted_spec.abs().clamp_min(1e-8).pow(power)
    target_mag = target_spec.abs().clamp_min(1e-8).pow(power)
    magnitude_loss = torch.sum((pred_mag - target_mag).square())

    complex_loss = torch.sum((predicted_spec.real - target_spec.real).square())
    complex_loss += torch.sum((predicted_spec.imag - target_spec.imag).square())
    return magnitude_loss + complex_loss

## Set up layers

In [8]:
def channel_shuffle(x: Tensor, groups: int) -> Tensor:
    b, t, d = x.shape
    if d % groups != 0:
        raise ValueError("Feature dimension must be divisible by groups")

    features_per_group = d // groups
    x = x.view(b, t, groups, features_per_group)
    x = x.transpose(2, 3).contiguous()
    x = x.view(b, t, d)

    return x

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, kernel_size: Tuple[int, int] = (3, 2), stride: Tuple[int, int] = (1, 1)) -> None:
        super().__init__()

        kt, kf = kernel_size
        self.pad = (
            kf // 2, # left
            kf - 1 - kf // 2, # right
            kt // 2, # top
            kt // 2, # bottom
        )

        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
            groups=in_channels,
            bias=False
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU()
    def forward(self, x: Tensor) -> Tensor:
        x = F.pad(x, self.pad)
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        x = self.act(x)
        return x
    
class SeparableTConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, scale_factor: int = 2) -> None:
        super().__init__()
        self.scale_factor = scale_factor

        self.conv = SeparableConv2d(in_channels, out_channels, kernel_size=(3, 2))

    def forward(self, x: Tensor) -> Tensor:
        x = F.interpolate(
            x, 
            scale_factor=(1, self.scale_factor), # [width, height] ~ [time, frequency]
            mode="nearest"
        )
        return self.conv(x)

class GroupedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()

        if in_features % groups:
            raise ValueError("in_features must be divisible by groups")

        if out_features % groups:
            raise ValueError("out_features must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.in_per_group = in_features // groups
        self.out_per_group = out_features // groups

        self.layers = nn.ModuleList(
            [
                nn.Linear(
                    self.in_per_group,
                    self.out_per_group,
                )
                for _ in range(groups)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, T, D]

        chunks = x.split(self.in_per_group, dim=-1)

        output_chunks = []

        for layer, chunk in zip(self.layers, chunks):
            y = layer(chunk)
            output_chunks.append(y)

        x = torch.cat(output_chunks, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRU(nn.Module):
    def __init__(self, input_size: int = 512, hidden_size: int = 512, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()
        if input_size % groups:
            raise ValueError("input_size must be divisible by groups")
        if hidden_size % groups:
            raise ValueError("hidden_size must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.input_per_group = input_size // groups
        self.hidden_per_groups = hidden_size // groups

        self.grus = nn.ModuleList(
            [
                nn.GRU(
                    input_size=self.input_per_group,
                    hidden_size=self.hidden_per_groups,
                    batch_first=True,
                )
                for _ in range(groups)
            ]
        )


    def forward(self, x: Tensor) -> Tensor:
        # [B, T, D]
        chunks = x.split(self.input_per_group, dim=-1)
        outputs = []

        for gru, chunk in zip(self.grus, chunks):
            # GRU returns [output, hidden]
            y, _ = gru(chunk)
            outputs.append(y)

        x = torch.cat(outputs, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRUStack(nn.Module):
    def __init__(self, size: int = 512, groups: int = 8, num_layers: int = 3) -> None:
        super().__init__()

        self.layers = nn.ModuleList(
            [
                GroupedGRU(
                    input_size=size,
                    hidden_size=size,
                    groups=groups,
                    shuffle=True,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers:
            x = layer(x)

        return x

class PConv(nn.Module):
    def __init__(self, channels: int = 64) -> None:
        super().__init__()

        self.conv = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x)

## Implement ERB encoder, decoder and deep filter net

In [9]:
class ERBEncoder(nn.Module):
    def __init__(self, erb_bins: int = 32, channels: int = 64, hidden_size: int = 512, groups: int = 8) -> None:
        super().__init__()

        if erb_bins % 8:
            raise ValueError("erb_bins must be divisible by 8")

        self.erb_bins = erb_bins
        self.channels = channels
        self.conv0 = SeparableConv2d(1, channels)
        self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2)) # B -> B / 2
        self.conv2 = SeparableConv2d(channels, channels, stride=(1, 2))
        self.conv3 = SeparableConv2d(channels, channels, stride=(1, 2))
        self.glinear = GroupedLinear(in_features=channels*erb_bins//8, out_features=hidden_size, groups=groups)
        self.gru = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=3)

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
        x0 = self.conv0(x)
        x1 = self.conv1(x0)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)

        batch, channels, time, freq = x3.shape
        x = x3.permute(0, 2, 1, 3)
        x = x.reshape(batch, time, channels*freq)

        x = self.glinear(x)
        embedding = self.gru(x)

        return x0, x1, x2, x3, embedding

class ERBDecoder(nn.Module):
    def __init__(self, channels: int = 64, erb_bins: int = 32, hidden_size: int = 512) -> None:
        super().__init__()

        bottleneck_freq = erb_bins // 8
        bottleneck_size = channels * bottleneck_freq

        self.channels = channels
        self.bottelneck_freq = bottleneck_freq

        self.linear = GroupedLinear(hidden_size, bottleneck_size)

        self.p3 = PConv(channels)
        self.p2 = PConv(channels)
        self.p1 = PConv(channels)
        self.p0 = PConv(channels)

        self.up3 = SeparableTConv2d(channels, channels)
        self.up2 = SeparableTConv2d(channels, channels)
        self.up1 = SeparableTConv2d(channels, channels)

        self.final_conv = SeparableConv2d(channels, 1, kernel_size=(3, 2))
        self.output_activation = nn.Sigmoid()

    def forward(self, embedding: Tensor, x0: Tensor, x1: Tensor, x2: Tensor, x3: Tensor) -> Tensor:
        batch, time, _ = embedding.shape

        x = self.linear(embedding)

        x = x.reshape(batch, time, self.channels, self.bottelneck_freq)
        x = x.permute(0, 2, 1, 3)

        x = x + self.p3(x3)

        x = self.up3(x)
        x = x + self.p2(x2)

        x = self.up2(x)
        x = x + self.p1(x1)

        x = self.up1(x)
        x = x + self.p0(x0)

        gains = self.final_conv(x)
        gains = self.output_activation(gains)

        return gains

# class DFNet(nn.Module):
#     def __init__(self, df_bins: int, df_order: int, channels: int=64, hidden_size: int = 512, groups: int=8) -> None:
#         super().__init__()

#         self.df_bins = df_bins
#         self.df_order = df_order
#         self.channels = channels

#         self.conv0 = SeparableConv2d(1, channels, stride=(1, 1))
#         self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2))
#         self.projection = GroupedLinear(channels*(df_bins // 2), hidden_size, groups=groups)
#         self.grus = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=2)
#         self.pconv = PConv(channels)
#         self.output = nn.Linear(hidden_size, df_bins*df_order*2)
#         self.merge = nn.Linear(hidden_size*2, hidden_size)

#     def forward(self, complex_features: Tensor, encoder_embedding: Tensor) -> Tensor:
#         x = self.conv0(complex_features)
#         x = self.conv1(x)

#         batch, channels, time, freq = x.shape
#         x = x.permute(0, 2, 1, 3)
#         x = x.reshape(batch, time, channels * freq)

#         x = self.projection(x)
#         x = self.grus(x)

#         x = torch.cat([x, encoder_embedding], dim=-1)
#         x = self.merge(x)

#         coefficients = self.output(x)

#         coefficients = coefficients.reshape(batch, time, self.df_bins, self.df_order, 2)

#         return coefficients

        

## Deep Neural Network (DNN)

In [10]:
class DeepFilterNetDNN(nn.Module):
    """
    rb_features:
        [B, 1, T, B_erb]

    complex_features:
        [B, 1, T, F_df]

    erb_gains:
        [B, 1, T, B_erb]
    
    df_coefficients:
        [B, T, F_df, N, 2]
    """

    def __init__(self, erb_bins: int = 32, df_bins: int = 96, df_order: int = 5, channels: int = 64, hidden_size: int = 512, groups: int = 8) -> None:
        super().__init__()

        self.encoder = ERBEncoder(
            erb_bins=erb_bins,
            channels=channels,
            hidden_size=hidden_size,
            groups=groups
        )

        self.decoder = ERBDecoder(
            channels=channels,
            erb_bins=erb_bins,
            hidden_size=hidden_size
        )

        # self.df_net = DFNet(
        #     df_bins=df_bins,
        #     df_order=df_order,
        #     channels=channels,
        #     hidden_size=hidden_size,
        #     groups=groups,
        # )

    # def forward(self, erb_features: Tensor, complex_features: Tensor) -> Tuple[Tensor, Tensor]:
    #     e0, e1, e2, e3, embedding = self.encoder(erb_features)
    #     gains = self.decoder(embedding, e0, e1, e2, e3)
    #     #df_coefficients = self.df_net(complex_features, embedding)

    #     return gains, df_coefficients

    def forward(self, erb_features: Tensor) -> Tensor:
            e0, e1, e2, e3, embedding = self.encoder(erb_features)
            gains = self.decoder(embedding, e0, e1, e2, e3)
            #df_coefficients = self.df_net(complex_features, embedding)
            return gains

## Generate erb band data and index files

In [11]:
input_dir = Path("/content/AEC-Challenge/datasets/synthetic/nearend_mic_signal")

target_dir = Path("/content/AEC-Challenge/datasets/synthetic/nearend_speech")

sample_rate = 16000
n_fft = 512
hop_length = 128
win_length = 512
erb_bins = 32
batch_size = 8
epochs = 10
learning_rate = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

filterbank = make_erb_filterbank(sample_rate=sample_rate, n_fft=n_fft, erb_bins=erb_bins)
build_erb_store(input_dir=input_dir, target_dir=target_dir, output_dir="/content/erb_store", 
                                        sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, win_length=win_length, erb_bins=erb_bins, device=device)



Processing 0/10000
Processing 10/10000
Processing 20/10000
Processing 30/10000
Processing 40/10000
Processing 50/10000
Processing 60/10000
Processing 70/10000
Processing 80/10000
Processing 90/10000
Processing 100/10000
Processing 110/10000
Processing 120/10000
Processing 130/10000
Processing 140/10000
Processing 150/10000
Processing 160/10000
Processing 170/10000
Processing 180/10000
Processing 190/10000
Processing 200/10000
Processing 210/10000
Processing 220/10000
Processing 230/10000
Processing 240/10000
Processing 250/10000
Processing 260/10000
Processing 270/10000
Processing 280/10000
Processing 290/10000
Processing 300/10000
Processing 310/10000
Processing 320/10000
Processing 330/10000
Processing 340/10000
Processing 350/10000
Processing 360/10000
Processing 370/10000
Processing 380/10000
Processing 390/10000
Processing 400/10000
Processing 410/10000
Processing 420/10000
Processing 430/10000
Processing 440/10000
Processing 450/10000
Processing 460/10000
Processing 470/10000
Pro

In [ ]:
# import shutil
# from pathlib import Path

# erb_store = Path("/content/erb_store")

# if erb_store.exists():
#     shutil.rmtree(erb_store)

# print("Deleted:", not erb_store.exists())

Deleted: True


In [29]:
!cp -r /content/erb_store /content/drive/MyDrive/

cp: error writing '/content/drive/MyDrive/erb_store/shard_0002.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0003.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0004.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0005.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0006.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0007.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0008.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0009.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0010.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0011.pt': No space left on device
cp: error writing '/content/drive/MyDrive/erb_store/shard_0012.pt': No space lef

In [2]:
!ls /content

sample_data


## Training Step

In [ ]:
indx_path = "/content/drive/MyDrive/erb_store/index.jsonl"
dataset = IndexedERBDataset(index_path=indx_path)

model = DeepFilterNetDNN(erb_bins=erb_bins).to(device)
synthesis_matrix = erb_synthesis_matrix(filterbank).to(device)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

## Check whether GPU is available or not

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Test DNN

In [ ]:
sample_rate = 16000
n_fft = 512
hop_length = 128
win_length = 512
erb_bins = 32

filterbank = make_erb_filterbank(
    sample_rate=sample_rate,
    n_fft=n_fft,
    erb_bins=erb_bins,
)

input_erb = wav_to_erb(
    "/content/drive/MyDrive/nearend_mic_fileid_0.wav",
    sample_rate,
    filterbank,
    n_fft,
    hop_length,
    win_length,
)

target_erb = wav_to_erb(
    "/content/drive/MyDrive/nearend_speech_fileid_0.wav",
    sample_rate,
    filterbank,
    n_fft,
    hop_length,
    win_length,
)

num_frames = min(input_erb.shape[0], target_erb.shape[0])

input_erb = input_erb[:num_frames]
target_erb = input_erb[:num_frames]

# Add batch and channel dimensions:
# [T, E] -> [B, 1, T, E]
input_features = input_erb.unsqueeze(0).unsqueeze(0)
target_features = target_erb.unsqueeze(0).unsqueeze(0)


target_mask = target_features / (input_features + 1e-8)
target_mask = target_mask.clamp(0.0, 1.0)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepFilterNetDNN(erb_bins=erb_bins, channels=64, hidden_size=512, groups=8).to(device)

model.eval()

input_features = input_features.to(device)
target_mask = target_mask.to(device)

with torch.no_grad():
    predicted_gains = model(input_features)


print("Input:", input_features.shape)
print("Predicted gains:", predicted_gains.shape)
print("Target mask:", target_mask.shape)
print(input_features)
print(predicted_gains)

In [ ]:
wav, sr = load_mono("/content/drive/MyDrive/nearend_speech_fileid_0.wav")

print(wav)
print("Sample rate:", sr)
print("Number of samples:", wav.shape)
print("Duration:", wav.shape[0] / sr, "seconds")

magnitude = stft_magnitude(
    wav,
    n_fft=512,
    hop_length=128,
    win_length=512,
)

print("STFT shape:", magnitude.shape)